# Phase 2: Spatiotemporal Error Analysis & Graph Model Diagnostics

**Author:** Member 4 (`phase2/m4-evaluation`)  
**Project:** METR-LA Highway Traffic Speed Forecasting — Phase 2 Deep & Graph Benchmark  

This notebook evaluates multi-horizon traffic speed forecasts, isolating where and when road network graph representations (GAT, DCRNN) provide statistically measurable advantages over non-graph deep controls (`gru_local`, `gat_identity`, `dcrnn_identity`, `gat_shuffled`) and Phase 1 competitive tree baselines (`xgboost_ha`).

> **Scientific Disclaimers & Terminology Invariants:**
> - Speeds are measured in **source speed units** (unverified, likely mph).
> - Timestamps represent the **stored clock** (unverified timezone).
> - Spatial groupings represent deterministic **coordinate regions**, not administrative freeways.
> - Congestion states are identified via a declared **proxy threshold** (< 60% of training 85th percentile speed).


In [ ]:
import os
import sys
from pathlib import Path

# Ensure writable matplotlib cache
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib_p2"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add repository root to path
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

METADATA_DIR = REPO_ROOT / "data" / "metadata" / "phase2"
FIGURES_DIR = REPO_ROOT / "figures" / "phase2"

print(f"Repository Root: {REPO_ROOT}")
print(f"Metadata Dir:    {METADATA_DIR}")


## 1. Overall Forecast Performance Across Horizons (5, 15, 30, 60 min)

We load the aggregated metrics summary across all seeds on the held-out test split.


In [ ]:
summary_path = METADATA_DIR / "metrics_summary.csv"
if summary_path.is_file():
    df_summary = pd.read_csv(summary_path)
    test_all = df_summary[(df_summary["split"] == "test") & (df_summary["period"] == "all")]
    pivot_mae = test_all.pivot(index="model", columns="horizon", values="mae_mean")[["5min", "15min", "30min", "60min"]]
    display(pivot_mae.style.format("{:.3f}").highlight_min(color="lightgreen", axis=0))
else:
    print(f"Notice: {summary_path.name} not found. Run scripts/phase2_eval_all.py to generate tables.")


## 2. Rush Hour vs. Off-Peak Error Stratification

Non-recurrent congestion during peak hours (07:00–10:00 and 16:00–19:00 on weekdays) presents the primary forecasting challenge. We examine whether graph models mitigate error spikes during peak congestion.


In [ ]:
if summary_path.is_file():
    peak_comp = df_summary[
        (df_summary["split"] == "test")
        & (df_summary["horizon"] == "60min")
        & (df_summary["period"].isin(["peak", "off_peak"]))
    ]
    if not peak_comp.empty:
        pivot_peak = peak_comp.pivot(index="model", columns="period", values="mae_mean")[["peak", "off_peak"]]
        pivot_peak["peak_increase_%"] = (pivot_peak["peak"] - pivot_peak["off_peak"]) / pivot_peak["off_peak"] * 100
        display(pivot_peak.style.format({"peak": "{:.3f}", "off_peak": "{:.3f}", "peak_increase_%": "+{:.1f}%"}))


## 3. Graph Architecture & Topology Ablations

To isolate the genuine value of road network topology, we compare:
1. **GAT (Bidirectional)**: GAT with real highway adjacency.
2. **GAT Identity (Control)**: GAT architecture with self-loops only (isolating temporal attention without spatial neighbours).
3. **GAT Shuffled (Control)**: GAT with permuted sensor labels (preserving node degree multiset while destroying spatial connectivity).


In [ ]:
fig_ablation_path = FIGURES_DIR / "fig_graph_ablation.png"
if fig_ablation_path.is_file():
    from IPython.display import Image
    display(Image(filename=str(fig_ablation_path)))
else:
    print("Graph ablation figure pending evaluation runner.")


## 4. Formal Hypothesis Testing (H0–H5)

Evaluated via paired circular moving-block bootstrap (2-day blocks, 2000 resamples, seed 42).


In [ ]:
hyp_path = METADATA_DIR / "hypotheses.csv"
if hyp_path.is_file():
    df_hyp = pd.read_csv(hyp_path)
    display(df_hyp[["id", "statement", "comparison", "mae_difference", "verdict"]])
else:
    print("Hypotheses table not found. Run scripts/phase2_eval_all.py to compute verdicts.")


## 5. Congestion Onset Detection Diagnostics

Evaluating precision and recall of predicting proxy congestion transitions (speed dropping below 60% of sensor free-speed).


In [ ]:
events_path = METADATA_DIR / "events.csv"
if events_path.is_file():
    df_events = pd.read_csv(events_path)
    display(df_events[["model", "horizon", "precision_congestion", "recall_congestion", "recall_congestion_onset"]])
else:
    print("Events diagnostics table not found.")
